# BPMN Assistant - Stage 2: DPO [Kaggle]

Preference-aligns the **Stage-1 SFT adapter** on `data/preference/*`. Fresh kernel = clean 16 GB GPU.

## Before running
1. **Add Input -> Notebook Output** = your committed Stage-1 SFT notebook (provides `bpmn-sft-adapter`).
2. Also attach the **bpmn-training-data** dataset (for `dpo_train.jsonl`).
3. Accelerator = **GPU T4 x2**, Internet = **On**. Run All.

`DATA_DIR` and `SFT_ADAPTER` are auto-detected. Download `bpmn-dpo-adapter.zip` at the end - final model.

In [ ]:
# 1) Dependencies
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use ONE T4 - stops HF Trainer DataParallel across both GPUs (OOM cause)
!pip install -q -U "transformers>=4.45,<5.0" "trl==0.21.0" "peft>=0.13" "datasets>=2.20" "bitsandbytes>=0.44" "accelerate>=1.0"
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    print("No HF_TOKEN secret - continuing unauthenticated (fine for public Qwen).")

In [ ]:
# 2) Config + auto-detect data and SFT adapter
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"          # MUST match Stage 1
OUTPUT_DIR = "/kaggle/working"
MAX_LEN    = 1024
ENABLE_THINKING = False
DPO_EPOCHS = 1
SYSTEM_PROMPT = 'You are a BPMN 2.0 expert assistant. Answer precisely and follow BPMN 2.0 conventions. When asked to generate a diagram, output valid BPMN 2.0 XML. When asked to review a diagram, identify concrete issues and how to fix them.'
import os, glob
_d = glob.glob("/kaggle/input/**/dpo_train.jsonl", recursive=True)
_a = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert _d, "dpo_train.jsonl not found - attach bpmn-training-data."
assert _a, "SFT adapter not found - Add Input -> Notebook Output of Stage 1."
DATA_DIR = os.path.dirname(_d[0]); SFT_ADAPTER = os.path.dirname(_a[0])
print("DATA_DIR =", DATA_DIR); print("SFT_ADAPTER =", SFT_ADAPTER)

In [ ]:
# 3) Tokenizer + DPO data (prompt formatted with thinking OFF)
from transformers import AutoTokenizer
from datasets import load_dataset
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
dpo = load_dataset("json", data_files={"train": f"{DATA_DIR}/dpo_train.jsonl", "val": f"{DATA_DIR}/dpo_val.jsonl"})
def fmt(ex):
    p = tok.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":ex["prompt"]}],
                                tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    return {"prompt": p, "chosen": ex["chosen"], "rejected": ex["rejected"]}
dpo = dpo.map(fmt, remove_columns=[c for c in dpo["train"].column_names if c not in ("prompt","chosen","rejected")])

In [ ]:
# 4) Load base 4-bit on a SINGLE GPU + attach the SFT adapter (trainable policy)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
base.config.use_cache = False
model = PeftModel.from_pretrained(base, SFT_ADAPTER, is_trainable=True)   # ref = adapter-disabled base
model.enable_input_require_grads()

In [ ]:
# 5) DPO
from trl import DPOConfig, DPOTrainer
# Neutralize TRL's chunked-CE LM-head patch (crashes on 4-bit/device_map functools.partial forward);
# it is only a loss memory-optimization, so training is unchanged.
import trl.trainer.sft_trainer as _sftmod
try:
    import trl.trainer.dpo_trainer as _dpomod
except Exception:
    _dpomod = None
_noop = lambda *a, **k: None
for _mod in (_sftmod, _dpomod):
    if _mod is None: continue
    for _t in (_mod, getattr(_mod,"SFTTrainer",None), getattr(_mod,"DPOTrainer",None)):
        if _t is not None and hasattr(_t, "_patch_chunked_ce_lm_head"):
            setattr(_t, "_patch_chunked_ce_lm_head", _noop)
dpo_cfg = DPOConfig(output_dir=f"{OUTPUT_DIR}/dpo", per_device_train_batch_size=1, gradient_accumulation_steps=16,
                    num_train_epochs=DPO_EPOCHS, learning_rate=5e-6, beta=0.1, fp16=True, logging_steps=10,
                    save_strategy="steps", save_steps=50, save_total_limit=2, max_prompt_length=768, max_length=MAX_LEN, gradient_checkpointing=True,
                    gradient_checkpointing_kwargs={"use_reentrant": False}, optim="paged_adamw_8bit", report_to="none")
dpo_trainer = DPOTrainer(model=model, ref_model=None, args=dpo_cfg, train_dataset=dpo["train"],
                         eval_dataset=dpo["val"], processing_class=tok)
import os
from transformers.trainer_utils import get_last_checkpoint
_ck = get_last_checkpoint(dpo_cfg.output_dir) if os.path.isdir(dpo_cfg.output_dir) else None
print("Resuming from", _ck) if _ck else print("No checkpoint found - starting fresh.")
dpo_trainer.train(resume_from_checkpoint=_ck)
dpo_trainer.save_model(f"{OUTPUT_DIR}/bpmn-dpo-adapter")
tok.save_pretrained(f"{OUTPUT_DIR}/bpmn-dpo-adapter")
import shutil; shutil.make_archive(f"{OUTPUT_DIR}/bpmn-dpo-adapter", "zip", f"{OUTPUT_DIR}/bpmn-dpo-adapter")
print("DPO adapter saved + zipped -> download bpmn-dpo-adapter.zip. This is your final model.")

In [ ]:
# 6) Quick sanity check (OPTIONAL - the adapter is already saved+zipped by cell 5).
import torch, gc
gc.collect(); torch.cuda.empty_cache()
model.config.use_cache = True     # re-enable KV cache for generation (training disabled it)
model.eval()
def chat(msg, n=256):
    txt = tok.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":msg}],
                                  tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    ids = tok(txt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False, use_cache=True)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
print(chat('Generate a BPMN 2.0 diagram: an employee submits a timesheet, a manager approves or rejects it, and the employee is notified.'))